# Dewarp ECG - Reproduction de la methode PM Cardio (Brevet US20240144417A1)

Pipeline complet en 14 cellules avec verification visuelle a chaque etape majeure.

**Entree** : image ECG + modele U-Net entraine (detecte les intersections de grille)

**Sortie** : image ECG redressee (grille parfaitement reguliere)

## Plan

### Partie A : Setup + Detection
1. Imports + chargement modele + image + detection
2. Visualisation : points sur image

### Partie B : Gridder (reconstruction de la grille)
3. Preprocessing + k-NN voisins
4. Visualisation : 1 point + ses 20 voisins
5. Liste aretes (longueur, angle)
6. Visualisation : scatter plot Fig 2G
7. DBScan clustering + gaussiennes
8. Visualisation : clusters Fig 2H
9. Selection max-vraisemblance + construction carres
10. Visualisation : carres detectes
11. BFS + inference des trous
12. Visualisation : grille complete

### Partie C : Undistortion
13. Template + warp par maille
14. Visualisation : comparaison avant/apres

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 1 : Setup - imports, chargement modele, image, detection
# ═══════════════════════════════════════════════════════════════════
import os, sys, glob, io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
import segmentation_models_pytorch as smp
from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
import networkx as nx

%matplotlib inline

PROJECT_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from shared.npz_schema import load_unified_npz, load_unified

# ─── Parametres ───
SAMPLE_FILE = "ECG_033_723_p0_aug.webp"
IMAGE_DIR   = os.path.join(PROJECT_ROOT, "data", "output_augmentation", "images")
NPZ_DIR     = os.path.join(PROJECT_ROOT, "data", "output_augmentation", "labels")
# Modele GAUSSIEN (le meilleur) : masques heatmap gaussiens + centroide pondere
RUNS_DIR    = os.path.join(PROJECT_ROOT, "data", "training", "runs_npz_gaussienne")
MODEL_RES   = 1024   # taille d'entree du U-Net
NPZ_KEY     = "grid_major_5mm"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device: {DEVICE}")

# ─── Chargement du modele entraine (latest run) ───
runs = sorted(glob.glob(os.path.join(RUNS_DIR, "run_*")))
ckpt_path = os.path.join(runs[-1], "checkpoints", "best_model.pth")
print(f"[INFO] Checkpoint : {ckpt_path}")

model = smp.Unet(encoder_name="resnet34", encoder_weights=None,
                 in_channels=3, classes=1, activation=None)
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(DEVICE).eval()
print(f"[INFO] Modele charge (epoch {ckpt['epoch']}, val_dice {ckpt.get('val_dice', 'N/A')})")

# ─── Chargement image native ───
img_pil = Image.open(os.path.join(IMAGE_DIR, SAMPLE_FILE)).convert("RGB")
Wn, Hn = img_pil.size
img_native = np.array(img_pil)
print(f"[INFO] Image : {SAMPLE_FILE}  ({Wn}x{Hn})")

# ─── Detection des points via U-Net ───
img_resized = img_pil.resize((MODEL_RES, MODEL_RES), Image.BILINEAR)
img_np_norm = np.array(img_resized, dtype=np.float32) / 255.0
img_tensor  = torch.from_numpy(img_np_norm).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)

with torch.no_grad():
    pred_sigmoid = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()
pred_native = cv2.resize(pred_sigmoid, (Wn, Hn), interpolation=cv2.INTER_LINEAR)
pred_binary = (pred_native > 0.5).astype(np.uint8)

# Extraction des centroides (centroide pondere par l'intensite, comme PM Cardio)
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(pred_binary, connectivity=8)
MIN_BLOB_AREA = 3
points = []
for lab in range(1, num_labels):
    if stats[lab, cv2.CC_STAT_AREA] < MIN_BLOB_AREA:
        continue
    blob = (labels == lab)
    weights = pred_native[blob]
    yy, xx = np.where(blob)
    w_sum = weights.sum()
    if w_sum < 1e-6:
        continue
    points.append([float((xx * weights).sum() / w_sum),
                   float((yy * weights).sum() / w_sum)])
points = np.asarray(points)
print(f"[INFO] Points detectes : {len(points)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 2 : Verification visuelle - points detectes sur l'image
# ═══════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(1, 1, figsize=(18, 12))
ax.imshow(img_native)
ax.scatter(points[:, 0], points[:, 1], s=1, c="red", alpha=0.6, label=f"{len(points)} points")
ax.set_title(f"Detection initiale : {len(points)} intersections detectees", fontsize=12)
ax.legend(loc="upper right")
ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 3 : Gridder - Preprocessing (flip + rotation 45°) + k-NN
# ═══════════════════════════════════════════════════════════════════
# PM Cardio fait : flip vertical (axe X) puis rotation 45° clockwise.
# Le but : aligner les directions principales (haut/bas/gauche/droite)
# avec les axes du scatter plot (longueur, angle) pour mieux clusteriser.
# ────────────────────────────────────────────────────────────────────
K_NEIGHBORS = 20

# 1. Flip vertical : y -> Hn - y  (origine en haut-gauche -> origine en bas-gauche)
points_flipped = points.copy()
points_flipped[:, 1] = Hn - points_flipped[:, 1]

# 2. Rotation 45° clockwise : (x, y) -> ( (x+y)/sqrt(2), (-x+y)/sqrt(2) )
ANGLE = -np.pi / 4   # 45° clockwise
cos_a, sin_a = np.cos(ANGLE), np.sin(ANGLE)
R = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
points_prep = points_flipped @ R.T

print(f"[INFO] Points preprocesses : {points_prep.shape}")
print(f"       Range X : {points_prep[:, 0].min():.0f} -> {points_prep[:, 0].max():.0f}")
print(f"       Range Y : {points_prep[:, 1].min():.0f} -> {points_prep[:, 1].max():.0f}")

# 3. k-NN : pour chaque point, identifier ses 20 voisins les plus proches
tree = cKDTree(points_prep)
distances, neighbor_indices = tree.query(points_prep, k=K_NEIGHBORS + 1)
# La premiere colonne est le point lui-meme (distance 0) -> on l'exclut
neighbor_indices = neighbor_indices[:, 1:]
distances        = distances[:, 1:]
print(f"[INFO] Voisins identifies pour {len(points_prep)} points (k={K_NEIGHBORS})")
print(f"       Distance moyenne au plus proche voisin : {distances[:, 0].mean():.2f} px")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 4 : Verification visuelle - 1 point central + ses 20 voisins
# ═══════════════════════════════════════════════════════════════════
# On choisit un point central et on affiche ses 20 plus proches voisins
# (sur le repere PREPROCESSE, c'est-a-dire apres flip + rotation 45°).
# Le but : verifier qu'on a bien 20 voisins repartis autour, et qu'on
# voit emerger 4 directions principales (haut/bas/gauche/droite a 45°).
# ────────────────────────────────────────────────────────────────────
REF_IDX = len(points_prep) // 2   # un point central
ref_pt  = points_prep[REF_IDX]
nbrs    = points_prep[neighbor_indices[REF_IDX]]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Gauche : vue d'ensemble (tous les points en bleu + ref en rouge + voisins en vert)
axes[0].scatter(points_prep[:, 0], points_prep[:, 1], s=2, c="lightblue", alpha=0.5)
axes[0].scatter(nbrs[:, 0], nbrs[:, 1], s=40, c="green", label=f"{K_NEIGHBORS} voisins")
axes[0].scatter(ref_pt[0], ref_pt[1], s=80, c="red", marker="*", label="Reference")
axes[0].set_title("Nuage complet (espace preprocesse)")
axes[0].legend(); axes[0].axis("equal")

# Droite : zoom sur le point de reference
axes[1].scatter(ref_pt[0], ref_pt[1], s=200, c="red", marker="*", label="Reference")
for i, nb in enumerate(nbrs):
    axes[1].plot([ref_pt[0], nb[0]], [ref_pt[1], nb[1]], "g-", alpha=0.4, lw=0.8)
    axes[1].scatter(nb[0], nb[1], s=40, c="green")
axes[1].set_title(f"Zoom : point central + 20 aretes vers ses voisins")
axes[1].legend(); axes[1].axis("equal")
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 5 : Gridder - Liste des aretes (longueur + angle)
# ═══════════════════════════════════════════════════════════════════
# Chaque point genere 20 aretes (vers ses 20 voisins).
# Pour chaque arete : on calcule (longueur, angle).
# Resultat : tableau de ~50 000 aretes-comme-points dans l'espace
# (longueur, angle) qu'on va clusteriser.
# ────────────────────────────────────────────────────────────────────
N = len(points_prep)

# Vecteurs source -> voisins (shape : N x K x 2)
src = np.repeat(points_prep[:, None, :], K_NEIGHBORS, axis=1)  # (N, K, 2)
dst = points_prep[neighbor_indices]                            # (N, K, 2)
delta = dst - src                                              # (N, K, 2)

# Longueur et angle
edge_length = np.linalg.norm(delta, axis=2)                    # (N, K)
edge_angle  = np.arctan2(delta[..., 1], delta[..., 0])          # (N, K), entre -pi et +pi

# Aplatir pour avoir une liste (M, 2) ou M = N * K
edges_lenang = np.stack([edge_length.flatten(), edge_angle.flatten()], axis=1)

# Index (source, voisin) pour pouvoir retrouver l'arete physique apres
src_idx = np.repeat(np.arange(N), K_NEIGHBORS)
dst_idx = neighbor_indices.flatten()
edge_pairs = np.stack([src_idx, dst_idx], axis=1)              # (M, 2)

print(f"[INFO] {len(edges_lenang)} aretes generees (= {N} points x {K_NEIGHBORS} voisins)")
print(f"       Longueur : min={edges_lenang[:,0].min():.1f}, mean={edges_lenang[:,0].mean():.1f}, max={edges_lenang[:,0].max():.1f}")
print(f"       Angle    : [-π, π] (= [{-np.pi:.2f}, {np.pi:.2f}])")

# Normalisation : longueur / longueur_median * π pour avoir une echelle comparable aux angles
# (sinon l'axe longueur ecraserait l'axe angle dans DBScan)
length_median = np.median(edges_lenang[:, 0])
edges_lenang_norm = edges_lenang.copy()
edges_lenang_norm[:, 0] = edges_lenang[:, 0] / length_median * np.pi
print(f"[INFO] Longueurs normalisees : longueur_median = {length_median:.1f} px -> π")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 6 : Verification visuelle - Scatter plot des aretes (Fig 2G)
# ═══════════════════════════════════════════════════════════════════
# On veut VOIR les 4 clusters denses correspondant aux 4 directions
# de la grille (haut, bas, gauche, droite). Si l'image est bien
# preprocessee, ces clusters doivent apparaitre clairement a une
# longueur fixe (~ π apres normalisation) et a 4 angles separes de π/2.
# ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Gauche : echelle brute (longueur en pixels)
axes[0].scatter(edges_lenang[:, 0], edges_lenang[:, 1], s=0.5, alpha=0.3, c="black")
axes[0].set_xlabel("Longueur de l'arete (px)")
axes[0].set_ylabel("Angle de l'arete (rad)")
axes[0].set_title(f"Fig 2G - Aretes en (longueur, angle) brut\n{len(edges_lenang)} aretes")
axes[0].grid(True, alpha=0.3)
axes[0].set_yticks(np.arange(-np.pi, np.pi + 0.1, np.pi/4))
axes[0].set_yticklabels([f"-π", "-3π/4", "-π/2", "-π/4", "0", "π/4", "π/2", "3π/4", "π"])

# Droite : echelle normalisee (longueur ramenee a ~π pour comparable aux angles)
axes[1].scatter(edges_lenang_norm[:, 0], edges_lenang_norm[:, 1], s=0.5, alpha=0.3, c="black")
axes[1].set_xlabel("Longueur normalisee (px / longueur_mediane * π)")
axes[1].set_ylabel("Angle de l'arete (rad)")
axes[1].set_title("Fig 2G - Aretes en (longueur normalisee, angle)\nLes 4 directions principales doivent etre VISIBLES ici")
axes[1].grid(True, alpha=0.3)
axes[1].set_yticks(np.arange(-np.pi, np.pi + 0.1, np.pi/4))
axes[1].set_yticklabels([f"-π", "-3π/4", "-π/2", "-π/4", "0", "π/4", "π/2", "3π/4", "π"])
axes[1].axvline(np.pi, color="red", linestyle="--", alpha=0.4, label="longueur attendue (π)")
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 7 : Gridder - Filtrage par longueur + DBScan + gaussiennes
# ═══════════════════════════════════════════════════════════════════
# IMPORTANT : avec 20 voisins, on capture aussi des voisins diagonaux et
# eloignes qui creent des clusters supplementaires. On filtre d'abord par
# LONGUEUR pour ne garder que les aretes vers les 4 voisins cardinaux directs.
# ────────────────────────────────────────────────────────────────────
# 1. Estimer la longueur typique d'une arete cardinale (= plus proche voisin)
closest_dists = distances[:, 0]   # distance au plus proche voisin pour chaque point
L_typical = np.median(closest_dists)
print(f"[INFO] Longueur typique d'une arete cardinale : {L_typical:.1f} px (mediane des plus proches voisins)")

# 2. Filtrer les aretes : on ne garde que celles dont la longueur est proche de L_typical
LENGTH_TOLERANCE = 0.3   # +/- 30% de tolerance pour absorber la deformation
L_min = L_typical * (1 - LENGTH_TOLERANCE)
L_max = L_typical * (1 + LENGTH_TOLERANCE)
length_filter = (edges_lenang[:, 0] >= L_min) & (edges_lenang[:, 0] <= L_max)
print(f"[INFO] Filtre longueur : [{L_min:.1f}, {L_max:.1f}] px")
print(f"       Aretes conservees : {length_filter.sum()} sur {len(edges_lenang)} ({100*length_filter.mean():.1f}%)")

# Sous-ensemble d'aretes pour DBScan
edges_filtered = edges_lenang_norm[length_filter]
edge_pairs_filtered = edge_pairs[length_filter]
print(f"[INFO] DBScan sera lance sur {len(edges_filtered)} aretes filtrees")

# 3. DBScan sur les aretes filtrees
MIN_CLUSTER_SIZE = 500
DBSCAN_EPS = 0.20
DBSCAN_MIN_SAMPLES = 30

dbscan = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
cluster_labels_filtered = dbscan.fit_predict(edges_filtered)

unique, counts = np.unique(cluster_labels_filtered, return_counts=True)
print(f"[INFO] DBScan : {len(unique)-int(-1 in unique)} clusters trouves")

# 4. Garder les 4 plus gros clusters
order = np.argsort(-counts)
top_clusters = []
for idx in order:
    lab = unique[idx]
    if lab == -1:
        continue
    if counts[idx] < MIN_CLUSTER_SIZE:
        break
    top_clusters.append((lab, counts[idx]))
    if len(top_clusters) >= 4:
        break

print(f"[INFO] {len(top_clusters)} cluster(s) garde(s) (taille >= {MIN_CLUSTER_SIZE}) :")
for lab, c in top_clusters:
    mask = cluster_labels_filtered == lab
    mean_ang = edges_filtered[mask, 1].mean()
    print(f"       Cluster {lab}: {c} aretes, angle={mean_ang:+.2f} rad ({np.degrees(mean_ang):+.0f}°)")

# Tri par angle croissant
top_clusters.sort(key=lambda x: edges_filtered[cluster_labels_filtered == x[0], 1].mean())

# 5. Ajuster une gaussienne sur chaque cluster (sur les aretes filtrees)
gaussians = []
for lab, _ in top_clusters:
    mask = cluster_labels_filtered == lab
    data = edges_filtered[mask]
    mu  = data.mean(axis=0)
    cov = np.cov(data.T) + 1e-6 * np.eye(2)
    cov_inv = np.linalg.inv(cov)
    log_det = np.log(np.linalg.det(cov))
    gaussians.append((mu, cov_inv, log_det))

print(f"[INFO] {len(gaussians)} gaussiennes ajustees")

# 6. Application des gaussiennes a TOUTES les aretes (pas seulement filtrees)
# pour pouvoir classer toutes les aretes par direction.
def log_prob(edges, gauss):
    mu, cov_inv, log_det = gauss
    diff = edges - mu
    return -0.5 * (np.einsum("ij,jk,ik->i", diff, cov_inv, diff) + log_det)

edge_probs = np.stack([log_prob(edges_lenang_norm, g) for g in gaussians], axis=1)
edge_dir   = np.argmax(edge_probs, axis=1)

# Pour la visualisation : on garde aussi cluster_labels (pour TOUTES les aretes, pas seulement filtrees)
# en utilisant -1 (bruit) pour celles hors filtre, et le label DBScan pour les autres
cluster_labels = np.full(len(edges_lenang_norm), -1, dtype=int)
cluster_labels[length_filter] = cluster_labels_filtered

print(f"[INFO] Probabilites calculees pour {len(edges_lenang_norm)} aretes (totales)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 8 : Verification visuelle - Clusters identifies (Fig 2H)
# ═══════════════════════════════════════════════════════════════════
# On colore chaque arete-comme-point selon son cluster DBScan.
# On veut voir 4 zones de couleurs distinctes correspondant aux 4
# directions. Le bruit (label -1) est en gris pale.
# ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Gauche : tous les points colores par cluster
COLORS = ["red", "blue", "green", "orange"]
DIRECTION_NAMES = ["direction 1", "direction 2", "direction 3", "direction 4"]

# Bruit en gris pale d'abord
noise_mask = cluster_labels == -1
axes[0].scatter(edges_lenang_norm[noise_mask, 0], edges_lenang_norm[noise_mask, 1],
                s=0.5, alpha=0.15, c="lightgray", label=f"Bruit ({noise_mask.sum()})")

for i, (lab, count) in enumerate(top_clusters):
    mask = cluster_labels == lab
    color = COLORS[i % len(COLORS)]
    axes[0].scatter(edges_lenang_norm[mask, 0], edges_lenang_norm[mask, 1],
                    s=1.0, alpha=0.6, c=color,
                    label=f"Cluster {i+1} ({count} aretes)")
    # Ellipse representant la gaussienne ajustee (1 sigma)
    mu, cov_inv, _ = gaussians[i]
    cov = np.linalg.inv(cov_inv)
    eigvals, eigvecs = np.linalg.eigh(cov)
    angle_e = np.degrees(np.arctan2(eigvecs[1, -1], eigvecs[0, -1]))
    from matplotlib.patches import Ellipse
    ell = Ellipse(xy=mu, width=2*np.sqrt(eigvals[-1]), height=2*np.sqrt(eigvals[0]),
                  angle=angle_e, edgecolor=color, facecolor="none", lw=2)
    axes[0].add_patch(ell)

axes[0].set_xlabel("Longueur normalisee")
axes[0].set_ylabel("Angle (rad)")
axes[0].set_title(f"Fig 2H - {len(top_clusters)} clusters identifies (ellipses = gaussiennes 1σ)")
axes[0].legend(fontsize=8, loc="lower right")
axes[0].grid(True, alpha=0.3)

# Droite : aretes classifiees par direction la plus probable
for i in range(len(gaussians)):
    mask = edge_dir == i
    axes[1].scatter(edges_lenang_norm[mask, 0], edges_lenang_norm[mask, 1],
                    s=0.5, alpha=0.3, c=COLORS[i % len(COLORS)],
                    label=f"Dir {i+1} (max-likelihood: {mask.sum()})")
axes[1].set_xlabel("Longueur normalisee")
axes[1].set_ylabel("Angle (rad)")
axes[1].set_title("Classification de TOUTES les aretes par max-likelihood")
axes[1].legend(fontsize=8, loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 9 : Gridder - Selection max-vraisemblance + construction carres
# ═══════════════════════════════════════════════════════════════════
# Pour chaque point : on garde UNE seule arete par direction
# (celle avec la plus haute probabilite). Puis on cherche des carres
# fermes en tracant un chemin direction1 -> direction2 -> direction3 -> direction4.
# ────────────────────────────────────────────────────────────────────
N_DIRS = len(gaussians)

# 1. Pour chaque point, selectionner la meilleure arete par direction
# Reshape : edge_probs (M, N_DIRS) -> (N, K_NEIGHBORS, N_DIRS)
probs_per_point = edge_probs.reshape(N, K_NEIGHBORS, N_DIRS)

# Pour chaque (point, direction) : indice du voisin avec la prob maximale
best_neighbor_per_dir = probs_per_point.argmax(axis=1)         # (N, N_DIRS)
best_prob_per_dir     = probs_per_point.max(axis=1)            # (N, N_DIRS)

# Resultat : tableau (N, N_DIRS) qui donne, pour chaque point,
# l'INDICE absolu du voisin dans chacune des N_DIRS directions
best_neighbor_idx = neighbor_indices[np.arange(N)[:, None], best_neighbor_per_dir]   # (N, N_DIRS)

print(f"[INFO] {N_DIRS} arete(s) selectionnee(s) par point")

# 2. Construction des carres : pour chaque point P, suivre les 4 directions
# dans l'ordre (dir0 -> dir1 -> dir2 -> dir3) et verifier qu'on revient a P.
# Note : les directions sont triees par angle croissant, donc dir0 et dir2
# sont opposees, comme dir1 et dir3.
#
# Strategie simple : pour former un carre, on suit
#   P --(dir0)--> A --(dir1)--> B --(dir2)--> C --(dir3)--> P ?
# Si on revient a P, c'est un carre valide.
MIN_PROB_THRESHOLD = -10.0   # seuil min de log-prob acceptable

squares = []   # liste de tuples (P, A, B, C) en indices de points

if N_DIRS == 4:
    for p in range(N):
        a = best_neighbor_idx[p, 0]
        if a == p: continue
        b = best_neighbor_idx[a, 1]
        if b == a or b == p: continue
        c = best_neighbor_idx[b, 2]
        if c == b or c == a: continue
        d = best_neighbor_idx[c, 3]
        if d != p:
            continue
        # Carre ferme detecte. On verifie la qualite (proba min)
        min_log_prob = min(
            best_prob_per_dir[p, 0],
            best_prob_per_dir[a, 1],
            best_prob_per_dir[b, 2],
            best_prob_per_dir[c, 3],
        )
        if min_log_prob < MIN_PROB_THRESHOLD:
            continue
        squares.append((p, a, b, c))

print(f"[INFO] Carres detectes : {len(squares)}")
print(f"       ({len(squares)/N*100:.1f}% des points sont coin d'un carre valide)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 11 : Gridder - BFS plus grand reseau + assignation (row, col)
# + comblement par REGRESSION LINEAIRE LOCALE (brevet PM Cardio [0096],[0106])
# ═══════════════════════════════════════════════════════════════════
def inverse_preprocess(pts_prep):
    """Repasse de l'espace preprocesse (rotation+flip) vers le repere image."""
    R_inv = R.T
    pts_unrot = pts_prep @ R_inv.T
    pts_unflip = pts_unrot.copy()
    pts_unflip[:, 1] = Hn - pts_unflip[:, 1]
    return pts_unflip

points_img = inverse_preprocess(points_prep)

# 1. Graphe (nodes = points, edges = cotes des carres)
G = nx.Graph()
G.add_nodes_from(range(N))
for (p, a, b, c) in squares:
    G.add_edge(p, a, direction=0)
    G.add_edge(a, b, direction=1)
    G.add_edge(b, c, direction=2)
    G.add_edge(c, p, direction=3)

# 2. Plus grand composant connecte
components = list(nx.connected_components(G))
components.sort(key=len, reverse=True)
main_nodes = list(components[0])
print(f"[INFO] {len(components)} composante(s) connectee(s)")
print(f"       Plus grande : {len(main_nodes)} points ({len(main_nodes)/N*100:.1f}%)")
G_main = G.subgraph(main_nodes).copy()

# 3. BFS pour assigner (row, col)
direction_deltas = []
for d in range(N_DIRS):
    edge_vecs = []
    for u, v, data in G_main.edges(data=True):
        if data["direction"] == d:
            edge_vecs.append(points_prep[v] - points_prep[u])
        elif (data["direction"] + 2) % 4 == d:
            edge_vecs.append(points_prep[u] - points_prep[v])
    if edge_vecs:
        direction_deltas.append(np.median(edge_vecs, axis=0))

center_xy = points_prep[main_nodes].mean(axis=0)
dists_to_center = np.linalg.norm(points_prep[main_nodes] - center_xy, axis=1)
origin_node = main_nodes[np.argmin(dists_to_center)]

rc_map = {origin_node: (0, 0)}
queue = [origin_node]
delta_dir = {0: ( 0, +1), 1: (+1,  0), 2: ( 0, -1), 3: (-1,  0)}
while queue:
    u = queue.pop(0)
    r_u, c_u = rc_map[u]
    for v in G_main.neighbors(u):
        if v in rc_map:
            continue
        delta_uv = points_prep[v] - points_prep[u]
        best_d, best_score = 0, -np.inf
        for cand_d in range(4):
            ref = direction_deltas[cand_d] if cand_d < len(direction_deltas) else direction_deltas[cand_d % len(direction_deltas)]
            score = np.dot(delta_uv, ref) / (np.linalg.norm(delta_uv) * np.linalg.norm(ref) + 1e-6)
            if score > best_score:
                best_score, best_d = score, cand_d
        dr, dc = delta_dir[best_d]
        rc_map[v] = (r_u + dr, c_u + dc)
        queue.append(v)

rows = [r for r, c in rc_map.values()]
cols = [c for r, c in rc_map.values()]
r_min, r_max = min(rows), max(rows)
c_min, c_max = min(cols), max(cols)
print(f"[INFO] Grille assignee : rows = [{r_min}, {r_max}], cols = [{c_min}, {c_max}]")
print(f"       Taille grille : {r_max-r_min+1} x {c_max-c_min+1}")
print(f"       Points connus : {len(rc_map)}")

# 4. Comblement par REGRESSION LINEAIRE LOCALE (PM Cardio)
#    Pour chaque trou (r,c), on ajuste un modele affine local
#       x = a0 + a1*dr + a2*dc  ,  y = b0 + b1*dr + b2*dc
#    sur les ~10 noeuds connus les plus proches, puis on predit (r,c).
#    La regression EXTRAPOLE -> comble aussi les bords et les coins.
ROWS = list(range(r_min, r_max + 1))
COLS = list(range(c_min, c_max + 1))
full_grid_prep = np.full((len(ROWS), len(COLS), 2), np.nan, dtype=np.float64)
for nd, (r, c) in rc_map.items():
    full_grid_prep[r - r_min, c - c_min] = points_prep[nd]

n_missing_before = int(np.isnan(full_grid_prep[..., 0]).sum())
print(f"[INFO] Trous a combler : {n_missing_before}")

def _fit_predict(rc_known, xy_known, target):
    # Modele affine (x,y) = c0 + c1*dr + c2*dc ajuste par moindres carres
    dr = rc_known[:, 0] - target[0]
    dc = rc_known[:, 1] - target[1]
    A = np.stack([np.ones(len(dr)), dr, dc], axis=1)
    cx, *_ = np.linalg.lstsq(A, xy_known[:, 0], rcond=None)
    cy, *_ = np.linalg.lstsq(A, xy_known[:, 1], rcond=None)
    return np.array([cx[0], cy[0]])   # valeur au point cible (dr=dc=0)

# On comble par anneaux : a chaque passe on remplit les trous qui ont assez de
# voisins connus PROCHES, puis ces nouveaux points servent a la passe suivante.
for it in range(25):
    known = ~np.isnan(full_grid_prep[..., 0])
    miss = np.argwhere(~known)
    if len(miss) == 0:
        break
    kr, kc = np.where(known)
    if len(kr) < 4:
        break
    known_rc = np.stack([kr, kc], axis=1).astype(float)
    known_xy = full_grid_prep[known]
    tree = cKDTree(known_rc)
    new_grid = full_grid_prep.copy()
    progress = False
    # rayon qui s'elargit si on stagne (derniere passe = sans limite)
    radius = 2.5 if it < 18 else 1e9
    for (r_idx, c_idx) in miss:
        kq = min(10, len(known_rc))
        d, idx = tree.query([r_idx, c_idx], k=kq)
        d = np.atleast_1d(d); idx = np.atleast_1d(idx)
        near = idx[d <= radius]
        if len(near) < 4:
            continue
        new_grid[r_idx, c_idx] = _fit_predict(known_rc[near], known_xy[near], (r_idx, c_idx))
        progress = True
    full_grid_prep = new_grid
    if not progress and radius < 1e9:
        # rien rempli a ce rayon : forcer une passe sans limite au tour suivant
        continue

n_missing_after = int(np.isnan(full_grid_prep[..., 0]).sum())
print(f"[INFO] Trous restants apres comblement : {n_missing_after}")

shape_save = full_grid_prep.shape
full_grid_img = full_grid_prep.reshape(-1, 2)
mask_valid = ~np.isnan(full_grid_img[:, 0])
full_grid_img_back = full_grid_img.copy()
full_grid_img_back[mask_valid] = inverse_preprocess(full_grid_img[mask_valid])
full_grid_img = full_grid_img_back.reshape(shape_save)
print(f"[INFO] Grille finale dans le repere image : shape {full_grid_img.shape}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 13 : Undistortion - Template + warp par maille + fix rotation
# ═══════════════════════════════════════════════════════════════════
CELL_SIZE = 64

H_grid, W_grid = full_grid_img.shape[:2]
n_cells_rows = H_grid - 1
n_cells_cols = W_grid - 1
H_out = n_cells_rows * CELL_SIZE
W_out = n_cells_cols * CELL_SIZE
print(f"[INFO] Template : {n_cells_rows} x {n_cells_cols} mailles -> image {W_out} x {H_out} px")

undistorted = np.zeros((H_out, W_out, 3), dtype=np.uint8)

n_warped, n_skipped = 0, 0
for r in range(n_cells_rows):
    for c in range(n_cells_cols):
        src_corners = np.array([
            full_grid_img[r,     c    ],
            full_grid_img[r,     c + 1],
            full_grid_img[r + 1, c + 1],
            full_grid_img[r + 1, c    ],
        ], dtype=np.float32)
        if np.isnan(src_corners).any():
            n_skipped += 1
            continue
        dst_corners = np.array([
            [0, 0], [CELL_SIZE, 0], [CELL_SIZE, CELL_SIZE], [0, CELL_SIZE],
        ], dtype=np.float32)
        M = cv2.getPerspectiveTransform(dst_corners, src_corners)
        cell_warped = cv2.warpPerspective(
            img_native, np.linalg.inv(M),
            (CELL_SIZE, CELL_SIZE),
            flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE,
        )
        undistorted[r*CELL_SIZE:(r+1)*CELL_SIZE,
                    c*CELL_SIZE:(c+1)*CELL_SIZE] = cell_warped
        n_warped += 1

print(f"[INFO] {n_warped} mailles redressees, {n_skipped} ignorees (NaN)")

# ─── Fix rotation : si on a une image portrait alors qu'on attend du paysage, on rotate ───
if undistorted.shape[0] > undistorted.shape[1]:
    print(f"[FIX] Image en portrait ({undistorted.shape[1]}x{undistorted.shape[0]}) -> rotation 90° pour retrouver le paysage")
    undistorted = cv2.rotate(undistorted, cv2.ROTATE_90_COUNTERCLOCKWISE)

# === Correction d'orientation finale (le gridder a un repère arbitraire) ===
# ROTATION_FINALE = rotation horaire appliquée au résultat : 0 / 90 / 180 / 270.
# Idempotent : la cellule reconstruit `undistorted` à chaque exécution, la
# rotation n'est donc appliquée qu'une seule fois.
ROTATION_FINALE = 180
_rot_codes = {0: None, 90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}
if _rot_codes[ROTATION_FINALE] is not None:
    undistorted = cv2.rotate(undistorted, _rot_codes[ROTATION_FINALE])
    print(f"[FIX] Orientation finale : rotation {ROTATION_FINALE}deg")

print(f"[INFO] Image finale : {undistorted.shape[1]} x {undistorted.shape[0]}")

In [ ]:
# ───────────────────────────────────────────────────────────────────
# CELLULE 14 : Vérification visuelle finale - Avant / Après
# ───────────────────────────────────────────────────────────────────
# `undistorted` est déjà ré-orienté en CELLULE 13 (ROTATION_FINALE), donc
# les deux images sont dans le même sens et toutes les cellules suivantes aussi.
fig, axes = plt.subplots(1, 2, figsize=(22, 12))

axes[0].imshow(img_native)
axes[0].set_title(f"AVANT - Image originale ({Wn} x {Hn})", fontsize=13)
axes[0].axis("off")

axes[1].imshow(undistorted)
axes[1].set_title(f"APRES - Image redressee ({undistorted.shape[1]} x {undistorted.shape[0]})", fontsize=13)
axes[1].axis("off")

plt.tight_layout(); plt.show()

# Affichage du résultat seul, pleine résolution (pour zoom)
print()
print("Image redressee a resolution native (pour zoom inline) :")
buf = io.BytesIO()
Image.fromarray(undistorted).save(buf, format="PNG")
from IPython.display import display, Image as IPyImage
display(IPyImage(data=buf.getvalue(), format="png"))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 15 : Correspondance visuelle P1 (impression) vs dewarp
# ═══════════════════════════════════════════════════════════════════
# 2 images cote a cote :
#   1. P1 (output_impression/) - l'image source, deja parfaitement droite
#   2. P2 + DEWARP - notre resultat redresse a partir de l'image augmentee
#
# Si la methode marche, les deux images doivent avoir la meme structure
# (memes leads aux memes positions, meme orientation paysage).
# ────────────────────────────────────────────────────────────────────
P1_DIR = os.path.join(PROJECT_ROOT, "data", "output_impression", "images")
p1_filename = SAMPLE_FILE.replace("_aug.webp", ".webp")
p1_path     = os.path.join(P1_DIR, p1_filename)

if not os.path.exists(p1_path):
    print(f"[WARN] Image source P1 introuvable : {p1_path}")
else:
    img_p1 = np.array(Image.open(p1_path).convert("RGB"))
    print(f"[INFO] P1 impression : {p1_filename}  ({img_p1.shape[1]} x {img_p1.shape[0]})")
    print(f"[INFO] Dewarp        : {undistorted.shape[1]} x {undistorted.shape[0]}")

    fig, axes = plt.subplots(1, 2, figsize=(28, 12))
    axes[0].imshow(img_p1)
    axes[0].set_title(f"P1 - IMPRESSION (source non-augmentee)\n{img_p1.shape[1]} x {img_p1.shape[0]}", fontsize=14)
    axes[0].axis("off")

    axes[1].imshow(undistorted)
    axes[1].set_title(f"P2 + DEWARP (notre resultat)\n{undistorted.shape[1]} x {undistorted.shape[0]}", fontsize=14)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 16 : Le dewarp a-t-il bien remis l'image a plat ?
# ═══════════════════════════════════════════════════════════════════
# On mesure directement la grille de l'image de SORTIE :
#   1. Les lignes sont-elles DROITES ?  (horizontales a 0deg, verticales a 0deg)
#   2. L'espacement est-il REGULIER ?   (ecart-type des pas / pas median)
# Si oui -> la deformation (pli, perspective) a bien ete corrigee.
# ────────────────────────────────────────────────────────────────────
from scipy.signal import find_peaks

Hd, Wd = undistorted.shape[:2]

# Lignes de grille dans l'image de sortie (morphologie)
gray = cv2.cvtColor(undistorted, cv2.COLOR_RGB2GRAY)
thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY_INV, 21, 5)
h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (100, 1)))
v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 100)))

# Positions des lignes
h_peaks, _ = find_peaks(h_lines.sum(axis=1), height=h_lines.sum(axis=1).max() * 0.3, distance=20)
v_peaks, _ = find_peaks(v_lines.sum(axis=0), height=v_lines.sum(axis=0).max() * 0.3, distance=20)

# 1. REGULARITE : espacement entre lignes consecutives
h_sp = np.diff(h_peaks); v_sp = np.diff(v_peaks)
irreg_h = np.std(h_sp) / np.median(h_sp) * 100
irreg_v = np.std(v_sp) / np.median(v_sp) * 100

# 2. RECTITUDE : pente de chaque ligne (regression sur ses pixels)
def line_angles(peaks, lines_mask, axis, taille):
    angles = []
    for center in peaks:
        lo, hi = max(0, center - 3), min(taille, center + 4)
        band = lines_mask[lo:hi, :] if axis == 0 else lines_mask[:, lo:hi]
        ys, xs = np.where(band > 0)
        if len(xs) < 50:
            continue
        if axis == 0:   # ligne horizontale : y = a*x + b
            a, _ = np.polyfit(xs, ys + lo, 1)
        else:           # ligne verticale : x = a*y + b
            a, _ = np.polyfit(ys, xs + lo, 1)
        angles.append(np.degrees(np.arctan(a)))
    return np.asarray(angles)

ang_h = line_angles(h_peaks, h_lines, 0, Hd)
ang_v = line_angles(v_peaks, v_lines, 1, Wd)
max_skew = max(np.max(np.abs(ang_h)) if len(ang_h) else 0,
               np.max(np.abs(ang_v)) if len(ang_v) else 0)

# ─── Resultats ───
print("=" * 60)
print("LE DEWARP A-T-IL REMIS L'IMAGE A PLAT ?")
print("=" * 60)
print(f"  Lignes detectees : {len(h_peaks)} horizontales, {len(v_peaks)} verticales")
print(f"\n[REGULARITE de l'espacement]")
print(f"  Horizontal : pas median {np.median(h_sp):.1f} px, irregularite {irreg_h:.1f}%")
print(f"  Vertical   : pas median {np.median(v_sp):.1f} px, irregularite {irreg_v:.1f}%")
print(f"\n[RECTITUDE des lignes]")
print(f"  Biais max horizontal : {np.max(np.abs(ang_h)) if len(ang_h) else 0:.3f} deg  (attendu 0)")
print(f"  Biais max vertical   : {np.max(np.abs(ang_v)) if len(ang_v) else 0:.3f} deg  (attendu 0)")

ok = (irreg_h < 5) and (irreg_v < 5) and (max_skew < 0.5)
print("\n" + "=" * 60)
if ok:
    print("VERDICT : OUI - grille droite (<0.5deg) et reguliere (<5%)")
    print("          -> la deformation a bien ete corrigee.")
else:
    print("VERDICT : grille imparfaite, voir les valeurs ci-dessus.")
print("=" * 60)

# ─── Visualisation ───
fig, axes = plt.subplots(1, 2, figsize=(22, 9))

# Dewarp + grille parfaite verte par-dessus
overlay = undistorted.copy()
for x in range(0, Wd, CELL_SIZE):
    cv2.line(overlay, (x, 0), (x, Hd - 1), (0, 255, 0), 1)
for y in range(0, Hd, CELL_SIZE):
    cv2.line(overlay, (0, y), (Wd - 1, y), (0, 255, 0), 1)
axes[0].imshow(overlay)
axes[0].set_title("Dewarp + grille parfaite verte\nSi le vert suit le rose -> image bien a plat", fontsize=12)
axes[0].axis("off")

# Histogramme des biais d'angle
axes[1].hist(ang_h, bins=30, color="steelblue", alpha=0.7, label="Horizontales")
axes[1].hist(ang_v, bins=30, color="orange", alpha=0.7, label="Verticales")
axes[1].axvline(0, color="red", ls="--", label="parfait (0 deg)")
axes[1].set_title(f"Biais d'angle des lignes (max {max_skew:.3f} deg)\nPlus c'est centre sur 0, plus c'est droit", fontsize=12)
axes[1].set_xlabel("biais (deg)"); axes[1].legend(); axes[1].grid(alpha=0.25)

plt.tight_layout(); plt.show(); plt.close()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 17 : Comparaison grille DEWARP vs grille IMPRESSION (P1)
# (comparaison STRUCTURELLE - rappel : contaminee par le comblement,
#  a lire avec la cellule 16 qui est la vraie validation honnete)
# ═══════════════════════════════════════════════════════════════════
# Charge le NPZ de P1 (la cellule 16 ne le fait plus)
P1_DIR     = os.path.join(PROJECT_ROOT, "data", "output_impression", "images")
P1_NPZ_DIR = os.path.join(PROJECT_ROOT, "data", "output_impression", "labels")
p1_filename = SAMPLE_FILE.replace("_aug.webp", ".webp")
p1_path     = os.path.join(P1_DIR, p1_filename)
p1_npz_path = os.path.join(P1_NPZ_DIR, p1_filename.replace(".webp", ""))
data_p1   = load_unified(p1_npz_path)
gt_pts_p1 = data_p1.get(NPZ_KEY, np.empty((0, 2)))
img_p1    = np.array(Image.open(p1_path).convert("RGB"))
Hp, Wp    = img_p1.shape[:2]

# ─── 1. Grille du dewarp : connue par construction ───
nx_dewarp = Wd // CELL_SIZE
ny_dewarp = Hd // CELL_SIZE
spacing_dewarp = CELL_SIZE
print(f"[DEWARP]  Grille : {nx_dewarp} colonnes x {ny_dewarp} lignes")
print(f"          Espacement (X et Y) : {spacing_dewarp} px (par construction)")

# ─── 2. Grille P1 depuis NPZ ───
gt = gt_pts_p1
ys_sorted = np.sort(gt[:, 1]); xs_sorted = np.sort(gt[:, 0])
dy_all = np.diff(ys_sorted); dx_all = np.diff(xs_sorted)
dy_nonzero = dy_all[dy_all > 1.0]; dx_nonzero = dx_all[dx_all > 1.0]
spacing_p1_y = np.median(dy_nonzero); spacing_p1_x = np.median(dx_nonzero)
ny_p1 = 1 + int(round((ys_sorted[-1] - ys_sorted[0]) / spacing_p1_y))
nx_p1 = 1 + int(round((xs_sorted[-1] - xs_sorted[0]) / spacing_p1_x))
print(f"\n[P1]      Grille : {nx_p1} colonnes x {ny_p1} lignes")
print(f"          Espacement X : {spacing_p1_x:.2f} px  Y : {spacing_p1_y:.2f} px (mediane)")

print(f"\n[COMPARAISON STRUCTURELLE]")
print(f"  Colonnes : dewarp={nx_dewarp}  |  P1={nx_p1}  |  ecart={abs(nx_dewarp-nx_p1)}")
print(f"  Lignes   : dewarp={ny_dewarp}  |  P1={ny_p1}  |  ecart={abs(ny_dewarp-ny_p1)}")

# ─── 3. Affichage ───
fig, axes = plt.subplots(1, 2, figsize=(24, 10))
axes[0].imshow(img_p1)
axes[0].scatter(gt[:, 0], gt[:, 1], s=1, c='blue', alpha=0.6)
axes[0].set_title(f"P1 source ({Wp}x{Hp})\n{nx_p1} x {ny_p1} cellules, pas {spacing_p1_x:.0f} x {spacing_p1_y:.0f} px",
                  fontsize=12); axes[0].axis('off')

overlay = undistorted.copy()
for x in range(0, Wd, CELL_SIZE):
    cv2.line(overlay, (x, 0), (x, Hd - 1), (0, 255, 0), 1)
for y in range(0, Hd, CELL_SIZE):
    cv2.line(overlay, (0, y), (Wd - 1, y), (0, 255, 0), 1)
axes[1].imshow(overlay)
axes[1].set_title(f"Dewarp ({Wd}x{Hd}) + grille parfaite verte\n{nx_dewarp} x {ny_dewarp} cellules, pas {CELL_SIZE} px",
                  fontsize=12); axes[1].axis('off')

plt.tight_layout(); plt.show(); plt.close()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 18 : DEWARP d'une image RÉELLE (output_real) — fonction reutilisable
# ═══════════════════════════════════════════════════════════════════
# Tout le pipeline (detection -> gridder -> undistortion) dans UNE fonction,
# applicable a n'importe quelle photo reelle. Le gridder est invariant en
# rotation ; le repere de sortie est arbitraire -> on l'oriente avec
# rotation_out (0/90/180/270). rotation_in pre-tourne la photo si besoin.
# Reutilise model, MODEL_RES, DEVICE (cellule 1).
def dewarp_image(img_path, rotation_in=0, rotation_out=180, verbose=True):
    import networkx as nx
    img_pil = Image.open(img_path).convert("RGB")
    if rotation_in:
        img_pil = img_pil.rotate(-rotation_in, expand=True)
    Wn, Hn = img_pil.size
    img_native = np.array(img_pil)

    # Detection des intersections (centroide pondere)
    arr = np.array(img_pil.resize((MODEL_RES, MODEL_RES), Image.BILINEAR), dtype=np.float32) / 255.0
    t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
    with torch.no_grad():
        pred = torch.sigmoid(model(t)).squeeze().cpu().numpy()
    pred_native = cv2.resize(pred, (Wn, Hn), interpolation=cv2.INTER_LINEAR)
    pb = (pred_native > 0.5).astype(np.uint8)
    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(pb, connectivity=8)
    pts = []
    for lab in range(1, nlab):
        if stats[lab, cv2.CC_STAT_AREA] < 3:
            continue
        m = (labels == lab); w = pred_native[m]; yy, xx = np.where(m); s = w.sum()
        if s < 1e-6:
            continue
        pts.append([float((xx * w).sum() / s), float((yy * w).sum() / s)])
    points = np.asarray(pts)
    if len(points) < 50:
        print(f"[ECHEC] Trop peu de points detectes ({len(points)}).")
        return None, img_native, points, None
    if verbose:
        print(f"[INFO] {len(points)} intersections detectees")

    # Preprocessing (flip Y + rotation 45) + kNN
    K = 20
    pf = points.copy(); pf[:, 1] = Hn - pf[:, 1]
    ca, sa = np.cos(-np.pi / 4), np.sin(-np.pi / 4)
    Rm = np.array([[ca, -sa], [sa, ca]])
    pp = pf @ Rm.T
    tree = cKDTree(pp)
    dists, nbr = tree.query(pp, k=K + 1); nbr = nbr[:, 1:]; dists = dists[:, 1:]
    N = len(pp)

    # Aretes (longueur, angle) + normalisation
    delta = pp[nbr] - np.repeat(pp[:, None, :], K, axis=1)
    elen = np.linalg.norm(delta, axis=2); eang = np.arctan2(delta[..., 1], delta[..., 0])
    ela = np.stack([elen.flatten(), eang.flatten()], axis=1)
    lmed = np.median(ela[:, 0]); ela_n = ela.copy(); ela_n[:, 0] = ela[:, 0] / lmed * np.pi

    # Filtre longueur + DBSCAN + gaussiennes
    Lt = np.median(dists[:, 0]); lf = (ela[:, 0] >= Lt * 0.7) & (ela[:, 0] <= Lt * 1.3)
    ef = ela_n[lf]
    cl = DBSCAN(eps=0.20, min_samples=30).fit_predict(ef)
    uq, cnt = np.unique(cl, return_counts=True)
    top = []
    for idx in np.argsort(-cnt):
        if uq[idx] == -1:
            continue
        if cnt[idx] < 500:
            break
        top.append(uq[idx])
        if len(top) >= 4:
            break
    if len(top) < 4:
        print(f"[ECHEC] Seulement {len(top)} directions de grille trouvees (4 attendues).")
        return None, img_native, points, None
    top.sort(key=lambda L: ef[cl == L, 1].mean())
    gauss = []
    for L in top:
        d = ef[cl == L]; mu = d.mean(0); cov = np.cov(d.T) + 1e-6 * np.eye(2)
        gauss.append((mu, np.linalg.inv(cov), np.log(np.linalg.det(cov))))
    def lp(E, g):
        mu, ci, ld = g; df = E - mu
        return -0.5 * (np.einsum("ij,jk,ik->i", df, ci, df) + ld)
    eprob = np.stack([lp(ela_n, g) for g in gauss], axis=1)

    # Arete max-vraisemblance par direction + carres
    ND = 4
    ppd = eprob.reshape(N, K, ND); bnpd = ppd.argmax(1); bppd = ppd.max(1)
    bni = nbr[np.arange(N)[:, None], bnpd]
    squares = []
    for q in range(N):
        a = bni[q, 0]
        if a == q:
            continue
        b = bni[a, 1]
        if b == a or b == q:
            continue
        c = bni[b, 2]
        if c == b or c == a:
            continue
        if bni[c, 3] != q:
            continue
        if min(bppd[q, 0], bppd[a, 1], bppd[b, 2], bppd[c, 3]) < -10.0:
            continue
        squares.append((q, a, b, c))
    if verbose:
        print(f"[INFO] {len(squares)} carres detectes")
    if len(squares) < 10:
        print(f"[ECHEC] Trop peu de carres ({len(squares)}).")
        return None, img_native, points, None

    # BFS (row,col) + comblement par regression lineaire locale
    def inv_pre(P):
        Pu = P @ Rm; Pf = Pu.copy(); Pf[:, 1] = Hn - Pf[:, 1]; return Pf
    G = nx.Graph(); G.add_nodes_from(range(N))
    for (q, a, b, c) in squares:
        G.add_edge(q, a, direction=0); G.add_edge(a, b, direction=1)
        G.add_edge(b, c, direction=2); G.add_edge(c, q, direction=3)
    main = list(sorted(nx.connected_components(G), key=len, reverse=True)[0])
    Gm = G.subgraph(main).copy()
    ddir = []
    for dd in range(4):
        vs = []
        for u, v, da in Gm.edges(data=True):
            if da["direction"] == dd:
                vs.append(pp[v] - pp[u])
            elif (da["direction"] + 2) % 4 == dd:
                vs.append(pp[u] - pp[v])
        if vs:
            ddir.append(np.median(vs, axis=0))
    center = pp[main].mean(0); origin = main[np.argmin(np.linalg.norm(pp[main] - center, axis=1))]
    rc = {origin: (0, 0)}; bq = [origin]; deltad = {0: (0, 1), 1: (1, 0), 2: (0, -1), 3: (-1, 0)}
    while bq:
        u = bq.pop(0); ru, cu = rc[u]
        for v in Gm.neighbors(u):
            if v in rc:
                continue
            duv = pp[v] - pp[u]; bd, bs = 0, -np.inf
            for cd in range(4):
                ref = ddir[cd] if cd < len(ddir) else ddir[cd % len(ddir)]
                sc = np.dot(duv, ref) / (np.linalg.norm(duv) * np.linalg.norm(ref) + 1e-6)
                if sc > bs:
                    bs, bd = sc, cd
            dr, dcc = deltad[bd]; rc[v] = (ru + dr, cu + dcc); bq.append(v)
    rows = [r for r, _ in rc.values()]; cols = [c for _, c in rc.values()]
    rmin, rmax = min(rows), max(rows); cmin, cmax = min(cols), max(cols)
    full = np.full((rmax - rmin + 1, cmax - cmin + 1, 2), np.nan)
    for nd, (r, c) in rc.items():
        full[r - rmin, c - cmin] = pp[nd]
    known0 = ~np.isnan(full[..., 0]).copy()
    def fit(rck, xyk, tgt):
        dr = rck[:, 0] - tgt[0]; dc = rck[:, 1] - tgt[1]; A = np.stack([np.ones(len(dr)), dr, dc], 1)
        cx, *_ = np.linalg.lstsq(A, xyk[:, 0], rcond=None); cy, *_ = np.linalg.lstsq(A, xyk[:, 1], rcond=None)
        return np.array([cx[0], cy[0]])
    for it in range(25):
        known = ~np.isnan(full[..., 0]); miss = np.argwhere(~known)
        if len(miss) == 0:
            break
        kr, kc = np.where(known)
        if len(kr) < 4:
            break
        krc = np.stack([kr, kc], 1).astype(float); kxy = full[known]; kt = cKDTree(krc)
        ng = full.copy(); rad = 2.5 if it < 18 else 1e9
        for (ri, ci) in miss:
            kq = min(10, len(krc)); dd, idx = kt.query([ri, ci], k=kq)
            dd = np.atleast_1d(dd); idx = np.atleast_1d(idx); near = idx[dd <= rad]
            if len(near) < 4:
                continue
            ng[ri, ci] = fit(krc[near], kxy[near], (ri, ci))
        full = ng
    sh = full.shape; fi = full.reshape(-1, 2); mv = ~np.isnan(fi[:, 0])
    fb = fi.copy(); fb[mv] = inv_pre(fi[mv]); full_img = fb.reshape(sh)
    _inf = (~known0) & (~np.isnan(full[..., 0]))
    comble_pts = full_img[_inf]

    # Undistortion (warp par maille de 64 px)
    CS = 64; Hg, Wg = full_img.shape[:2]; nr, nc = Hg - 1, Wg - 1
    out = np.zeros((nr * CS, nc * CS, 3), dtype=np.uint8)
    for r in range(nr):
        for c in range(nc):
            scr = np.array([full_img[r, c], full_img[r, c + 1], full_img[r + 1, c + 1], full_img[r + 1, c]], dtype=np.float32)
            if np.isnan(scr).any():
                continue
            dst = np.array([[0, 0], [CS, 0], [CS, CS], [0, CS]], dtype=np.float32)
            M = cv2.getPerspectiveTransform(dst, scr)
            out[r * CS:(r + 1) * CS, c * CS:(c + 1) * CS] = cv2.warpPerspective(
                img_native, np.linalg.inv(M), (CS, CS), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    if out.shape[0] > out.shape[1]:
        out = cv2.rotate(out, cv2.ROTATE_90_COUNTERCLOCKWISE)
    codes = {0: None, 90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}
    if codes.get(rotation_out) is not None:
        out = cv2.rotate(out, codes[rotation_out])
    if verbose:
        print(f"[OK] Image redressee : {out.shape[1]}x{out.shape[0]}")
    return out, img_native, points, {"squares": squares, "main_nodes": set(main), "N": N, "comble_pts": comble_pts}

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 19 : Dewarp d'une image REELLE  ->  POINTS detectes + Avant/Apres
# ═══════════════════════════════════════════════════════════════════
%matplotlib inline
import glob
REAL_ROOT    = r"C:\Users\v\Desktop\ECGPerturb-main\data\output_real"
SUBFOLDER    = "photos_crumbles"   # ou None pour tout scanner
INDEX        = 1
ROTATION_IN  = 0     # pre-rotation de la photo (degres horaires) si de travers
ROTATION_OUT = 180   # orientation finale : 0 / 90 / 180 / 270

search = os.path.join(REAL_ROOT, SUBFOLDER if SUBFOLDER else "**", "*")
real_files = sorted([f for f in glob.glob(search, recursive=True)
                     if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp"))])
print(f"{len(real_files)} images reelles trouvees")
img_path = real_files[INDEX]
print(f"Image : {img_path}")

undist, original, points, _dbg = dewarp_image(img_path, rotation_in=ROTATION_IN, rotation_out=ROTATION_OUT)

# Affichage : GAUCHE = points detectes par le modele | DROITE = image redressee
fig, ax = plt.subplots(1, 2, figsize=(22, 11))
ax[0].imshow(original)
if points is not None and len(points):
    ax[0].scatter(points[:, 0], points[:, 1], s=1, c="lime", alpha=0.7)
nb_pts = 0 if points is None else len(points)
ax[0].set_title(f"POINTS detectes par le modele ({nb_pts})"); ax[0].axis("off")
if undist is not None:
    ax[1].imshow(undist)
    ax[1].set_title(f"APRES - redressee ({undist.shape[1]}x{undist.shape[0]})")
else:
    ax[1].text(0.5, 0.5, "Dewarp echoue\n(grille trop abimee sur cette photo)",
               ha="center", va="center", fontsize=16, color="red")
ax[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 20 : Visuel des POINTS detectes par le modele (image reelle)
# ═══════════════════════════════════════════════════════════════════
# Reutilise img_path + ROTATION_IN de la CELLULE 19. Montre les intersections
# detectees (centroide pondere) superposees sur la photo, + la heatmap brute.
%matplotlib inline
_pil = Image.open(img_path).convert("RGB")
if ROTATION_IN:
    _pil = _pil.rotate(-ROTATION_IN, expand=True)
_Wn, _Hn = _pil.size
_arr = np.array(_pil)
_in = np.array(_pil.resize((MODEL_RES, MODEL_RES), Image.BILINEAR), dtype=np.float32) / 255.0
_t = torch.from_numpy(_in).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
with torch.no_grad():
    _pred = torch.sigmoid(model(_t)).squeeze().cpu().numpy()
_pn = cv2.resize(_pred, (_Wn, _Hn), interpolation=cv2.INTER_LINEAR)
_pb = (_pn > 0.5).astype(np.uint8)
_nl, _lb, _st, _ = cv2.connectedComponentsWithStats(_pb, connectivity=8)
_pts = []
for _l in range(1, _nl):
    if _st[_l, cv2.CC_STAT_AREA] < 3:
        continue
    _m = (_lb == _l); _w = _pn[_m]; _yy, _xx = np.where(_m); _s = _w.sum()
    if _s < 1e-6:
        continue
    _pts.append([float((_xx * _w).sum() / _s), float((_yy * _w).sum() / _s)])
_pts = np.asarray(_pts)
print(f"{len(_pts)} points detectes  |  max heatmap = {_pn.max():.3f}")

fig, ax = plt.subplots(1, 2, figsize=(22, 11))
ax[0].imshow(_arr); ax[0].set_title(f"Photo reelle ({_Wn}x{_Hn})"); ax[0].axis("off")
ax[1].imshow(_arr)
if len(_pts):
    ax[1].scatter(_pts[:, 0], _pts[:, 1], s=1, c="lime", alpha=0.7)
ax[1].set_title(f"Points detectes par le modele ({len(_pts)})"); ax[1].axis("off")
plt.tight_layout(); plt.show()

# Heatmap brute du modele superposee (voir ou il "voit" la grille)
fig2, ax2 = plt.subplots(1, 1, figsize=(13, 9))
ax2.imshow(_arr)
ax2.imshow(_pn, cmap="hot", alpha=0.5, vmin=0, vmax=1)
ax2.set_title("Heatmap du modele (sigmoid) superposee"); ax2.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLULE 21 : Reseau de carres RETENU par le BFS (diagnostic)
# ═══════════════════════════════════════════════════════════════════
# VERT  = points/carres GARDES (plus grand reseau connecte -> utilises au dewarp)
# ROUGE = points EXCLUS (carres isoles, jetes par le BFS -> zone non redressee)
# Reutilise img_path / ROTATION_IN / ROTATION_OUT de la CELLULE 19.
%matplotlib inline
undist, original, points, dbg = dewarp_image(img_path, rotation_in=ROTATION_IN,
                                             rotation_out=ROTATION_OUT, verbose=False)
fig, axes = plt.subplots(1, 2, figsize=(24, 11))
ax = axes[0]
ax.imshow(original)
if dbg is not None and points is not None:
    main = dbg["main_nodes"]; squares = dbg["squares"]
    for (q, a, b, c) in squares:
        if q in main and a in main and b in main and c in main:
            poly = points[[q, a, b, c, q]]
            ax.plot(poly[:, 0], poly[:, 1], c="lime", lw=0.6, alpha=0.6)
    inmain = np.array([i in main for i in range(len(points))])
    if (~inmain).any():
        ax.scatter(points[~inmain, 0], points[~inmain, 1], s=1, c="red",  label=f"exclus ({int((~inmain).sum())})")
    ax.scatter(points[inmain, 0], points[inmain, 1], s=1, c="lime", label=f"reseau BFS ({int(inmain.sum())})")
    cp = dbg.get("comble_pts")
    if cp is not None and len(cp):
        ax.scatter(cp[:, 0], cp[:, 1], s=1, c="deepskyblue", label=f"combles ({len(cp)})")
    ax.legend(loc="upper right", fontsize=11)
    print(f"Points gardes (reseau): {int(inmain.sum())} | exclus: {int((~inmain).sum())} | carres: {len(squares)}")
else:
    ax.set_title("Gridder echoue (pas de reseau)")
ax.set_title("Points : vert=reseau garde | rouge=deconnecte (exclu) | bleu=comble (trou reconstruit)")
ax.axis("off")
if undist is not None:
    axes[1].imshow(undist); axes[1].set_title(f"APRES - redressee ({undist.shape[1]}x{undist.shape[0]})")
else:
    axes[1].text(0.5,0.5,"Dewarp echoue",ha="center",va="center",fontsize=16,color="red")
axes[1].axis("off")
plt.tight_layout(); plt.show()